In [3]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 指定zarr数组文件路径
zarr_array_path = "/home/lumina/lumina/Jiawei/RoboTwin/policy/DP/data/move_can_pot-demo_clean-50.zarr/data/head_camera"

def inspect_zarr_data(zarr_path):
    """
    检查Zarr数据文件的内容和格式
    """
    print("=== Zarr数据文件信息 ===")
    print(f"数据路径: {zarr_path}")
    
    # 打开zarr数组
    try:
        # 使用zarr.open_array打开数组
        data = zarr.open_array(zarr_path, mode='r')
        
        print(f"数组形状: {data.shape}")
        print(f"数据类型: {data.dtype}")
        print(f"块大小: {data.chunks}")
        print(f"压缩方式: {data.compressor}")
        print(f"填充值: {data.fill_value}")
        
        # 显示数组的基本统计信息
        print("\n=== 数据统计信息 ===")
        print(f"数据大小: {data.nbytes / (1024**2):.2f} MB")
        print(f"元素总数: {data.size}")
        
        # 如果数据不太大，显示一些样本
        print("\n=== 数据样本 ===")
        if data.shape[0] > 0:
            print("第一帧数据信息:")
            first_frame = data[0]  # 获取第一帧
            print(f"  帧形状: {first_frame.shape}")
            print(f"  数据类型: {first_frame.dtype}")
            print(f"  数值范围: [{first_frame.min()}, {first_frame.max()}]")
            print(f"  数据类型: {first_frame.dtype}")
            
            # 显示一些像素值样本
            print("  像素值样本 (前10个像素):")
            print(f"    R通道: {first_frame[0, :20, 0]}")  # 前20个R像素值
            print(f"    G通道: {first_frame[1, :20, 0]}")  # 前20个G像素值
            print(f"    B通道: {first_frame[2, :20, 0]}")  # 前20个B像素值
            
        # 显示几帧的统计信息
        num_frames_to_show = min(5, data.shape[0])
        print(f"\n=== 前{num_frames_to_show}帧统计信息 ===")
        for i in range(num_frames_to_show):
            frame = data[i]
            print(f"帧 {i}: 均值={frame.mean():.2f}, 最小值={frame.min()}, 最大值={frame.max()}")
            
    except Exception as e:
        print(f"读取zarr数组时出错: {e}")
        return None
    
    return data

def visualize_sample_frames(data, num_frames=3):
    """
    可视化几帧图像数据
    """
    if data is None:
        return
    
    num_frames = min(num_frames, data.shape[0])
    if num_frames == 0:
        print("没有数据可显示")
        return
    
    fig, axes = plt.subplots(1, num_frames, figsize=(15, 5))
    if num_frames == 1:
        axes = [axes]
    
    for i in range(num_frames):
        # 从CHW格式转换为HWC格式以便显示
        frame = data[i]  # [3, 240, 320]
        frame_rgb = np.transpose(frame, (1, 2, 0))  # [240, 320, 3]
        
        axes[i].imshow(frame_rgb)
        axes[i].set_title(f'Frame {i}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def analyze_data_structure(zarr_path):
    """
    分析数据结构
    """
    print("=== 数据结构分析 ===")
    
    # 检查.zarray文件
    zarray_file = Path(zarr_path) / ".zarray"
    if zarray_file.exists():
        with open(zarray_file, 'r') as f:
            import json
            zarray_info = json.load(f)
            print("Zarr元数据 (.zarray):")
            for key, value in zarray_info.items():
                print(f"  {key}: {value}")
    
    # 检查数据块
    print("\n=== 数据块信息 ===")
    try:
        data = zarr.open_array(zarr_path, mode='r')
        
        # 检查第一个数据块
        chunk_files = list(Path(zarr_path).glob("0.*"))
        if chunk_files:
            print(f"找到 {len(chunk_files)} 个数据块文件")
            print("示例块文件:")
            for i, chunk_file in enumerate(chunk_files[:5]):
                print(f"  {chunk_file.name}")
        
        return data
    except Exception as e:
        print(f"分析数据结构时出错: {e}")
        return None

# 主执行代码
if __name__ == "__main__":
    # 检查数据
    data = inspect_zarr_data(zarr_array_path)
    
    # 分析数据结构
    analyze_data_structure(zarr_array_path)
    
    # 可视化样本帧（可选）
    # visualize_sample_frames(data, num_frames=3)
    
    print("\n=== 数据格式总结 ===")
    print("根据.zarray文件信息:")
    print("- 形状: [7563, 3, 240, 320]")
    print("  - 7563: 帧数")
    print("  - 3: 颜色通道数 (RGB)")
    print("  - 240: 图像高度")
    print("  - 320: 图像宽度")
    print("- 数据类型: |u1 (8位无符号整数，即0-255范围的像素值)")
    print("- 块大小: [100, 3, 240, 320] (每块包含100帧)")

=== Zarr数据文件信息 ===
数据路径: /home/lumina/lumina/Jiawei/RoboTwin/policy/DP/data/move_can_pot-demo_clean-50.zarr/data/head_camera
数组形状: (7563, 3, 240, 320)
数据类型: uint8
块大小: (100, 3, 240, 320)
压缩方式: Blosc(cname='zstd', clevel=3, shuffle=SHUFFLE, blocksize=0)
填充值: 0

=== 数据统计信息 ===
数据大小: 1661.79 MB
元素总数: 1742515200

=== 数据样本 ===
第一帧数据信息:
  帧形状: (3, 240, 320)
  数据类型: uint8
  数值范围: [15, 255]
  数据类型: uint8
  像素值样本 (前10个像素):
    R通道: [232 235 232 232 232 229 227 226 226 225 225 225 228 229 230 232 230 232
 242 255]
    G通道: [217 220 217 217 216 213 211 210 210 209 209 209 210 211 212 214 214 218
 232 253]
    B通道: [215 218 215 215 217 214 212 211 211 210 210 210 209 210 211 213 215 219
 232 252]

=== 前5帧统计信息 ===
帧 0: 均值=232.58, 最小值=15, 最大值=255
帧 1: 均值=232.59, 最小值=18, 最大值=255
帧 2: 均值=232.59, 最小值=19, 最大值=255
帧 3: 均值=232.56, 最小值=15, 最大值=255
帧 4: 均值=232.56, 最小值=15, 最大值=255
=== 数据结构分析 ===
Zarr元数据 (.zarray):
  chunks: [100, 3, 240, 320]
  compressor: {'blocksize': 0, 'clevel': 3, 'cname': 'zstd', 'id':

In [1]:
import torch
import torch.nn.functional as F

# 生成随机张量
x = torch.randn(1, 1024, 24, 24)  # [B, C, H, W]
# 双线性插值到 [1, 1024, 240, 320]
x_resized = F.interpolate(x, size=(240, 320), mode='bilinear', align_corners=False)

print(x_resized.shape)  # 应输出: torch.Size([1, 1024, 240, 320])

torch.Size([1, 1024, 240, 320])


In [2]:
import os
import urllib.request
import torch
import clip
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import torch.nn.functional as F

class CLIP_encoder:
    def __init__(self, model_name="ViT-B/16", device=None):
        # import pdb; pdb.set_trace()
        self.device = device or torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        print(f"[INFO] Using device: {self.device}")

        self.model_dir = "/home/lumina/lumina/Jiawei/RoboTwin/Group3/models"
        os.environ["TORCH_HOME"] = self.model_dir
        self._ensure_model_download(model_name)

        self.model, self.preprocess = clip.load(model_name, device=self.device)
        

    def _ensure_model_download(self, model_name):
        model_url_map = {
            "ViT-L/14@336px": "https://openaipublic.azureedge.net/clip/models/3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02/ViT-L-14-336px.pt",
            "ViT-B/16": "https://openaipublic.azureedge.net/clip/models/5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f/ViT-B-16.pt"
        }
        model_file_map = {
            "ViT-L/14@336px": "ViT-L-14-336px.pt",
            "ViT-B/16": "ViT-B-16.pt"
        }

        if model_name not in model_file_map:
            raise ValueError(f"暂不支持模型: {model_name}")

        model_path = os.path.join(self.model_dir, "clip", model_file_map[model_name])
        if not os.path.exists(model_path):
            os.makedirs(os.path.dirname(model_path), exist_ok=True)
            print(f"[INFO] 模型未缓存，正在下载到: {model_path}")
            urllib.request.urlretrieve(model_url_map[model_name], model_path)
            print("[INFO] 模型下载完成")
        else:
            print(f"[INFO] 已检测到本地模型缓存：{model_path}")

    def Vit_336px_CLIPFeatureExtractor(self, image_path):
        assert os.path.exists(image_path), f"Image not found: {image_path}"
        image = Image.open(image_path).convert("RGB")
        image_input = self.preprocess(image).unsqueeze(0).to(self.device).to(torch.float16)
        print(f"[INFO] preprocess.shape: ", image_input.shape) # [1, 3, 336, 336]
        with torch.no_grad():
            _ = self.model.encode_image(image_input)
            visual = self.model.visual
            x = visual.conv1(image_input)  # shape: [1, 1024, 24, 24]
            print(f"[INFO] 提取的特征图形状: {x.shape}")
            return x
        
    def Vit_336px_CLIPFeatureExtractor_from_array(self, image_array: np.ndarray) -> np.ndarray:
        """
        输入：np.ndarray 图像，形状为 [H, W, 3]，dtype=uint8
        输出：降维后的 RGB 特征图，形状为 [3, 240, 320]
        """
        image = Image.fromarray(image_array).convert("RGB")
        image_input = self.preprocess(image).unsqueeze(0).to(self.device).to(torch.float16)

        with torch.no_grad():
            visual = self.model.visual
            feat = visual.conv1(image_input)  # [1, 1024, 24, 24]
            feat_resized = F.interpolate(feat, size=(240, 320), mode='bilinear', align_corners=False)
            feat_resized = feat_resized.squeeze(0).cpu().numpy()  # [1024, 240, 320]

            feat_flat = feat_resized.reshape(feat_resized.shape[0], -1).T  # [76800, 1024]
            pca = PCA(n_components=3)
            feat_pca = pca.fit_transform(feat_flat).T.reshape(3, 240, 320)  # [3, 240, 320]
            return feat_pca.astype(np.float32)


    def save_featmap_from_tensor(self, tensor, save_path1):
        if tensor.ndim == 3 and tensor.shape[0] == 1024:
            tensor = np.transpose(tensor, (1, 2, 0))

        if tensor.ndim == 3 and tensor.shape[2] == 3:
            tensor_min = tensor.min()
            tensor_max = tensor.max()
            tensor = (tensor - tensor_min) / (tensor_max - tensor_min + 1e-8)
            tensor = (tensor * 255).astype(np.uint8)

            os.makedirs(os.path.dirname(save_path1), exist_ok=True)
            Image.fromarray(tensor).save(save_path1)
            print(f"[INFO] 特征图已保存为 JPG 到: {save_path1}")
        else:
            print(f"[ERROR] 无法保存图像，张量的形状不符合要求: {tensor.shape}")

    def extract_feat_map(self, image_path):
        feat = self.Vit_336px_CLIPFeatureExtractor(image_path).detach().cpu()
        print(f"[DEBUG] 原始特征 shape: {feat.shape}")  # (1, 1024, 24, 24)
        # 双线性插值到 [1, 1024, 240, 320]
        feat_resized = F.interpolate(feat, size=(240, 320), mode='bilinear', align_corners=False)
        
        feat_resized = feat_resized.squeeze(0).numpy() # [1024, 240, 320]
        print(f"[DEBUG] 插值后的特征 shape: {feat_resized.shape}")
        C, H, W = feat_resized.shape  # [1024, 240, 320]
        # 展平空间维度，准备做 PCA：[H*W, C]
        feat_flat = feat_resized.reshape(C, -1).T  # [76800, 1024]
        
        pca = PCA(n_components=3)
        feat_resized_rgb = pca.fit_transform(feat_flat) # [76800, 3]
        feat_resized_rgb = feat_resized_rgb.T.reshape(3, H, W) # 转换为 [3, 240, 320]
        
        # 转置为 [H, W, C] → [240, 320, 3]
        feat_resized_rgb = np.transpose(feat_resized_rgb, (1, 2, 0))
        
        return feat_resized_rgb

    def save_feat_map(self, image_path, save_path1):
        feat_resized_rgb = self.extract_feat_map(image_path)
        self.save_featmap_from_tensor(feat_resized_rgb, save_path1)
   
    



    


In [4]:
from diffusion_policy.model.custom.CLIP_encoder import CLIP_encoder

encoder = CLIP_encoder()
print(encoder)


[INFO] Using device: cuda:0
[INFO] 已检测到本地模型缓存：/home/lumina/lumina/Jiawei/RoboTwin/Group3/models/clip/ViT-B-16.pt
